# Wedge Inspection — Outcome-Agreer Factor-Support Overlap

Loads a wedge run jsonl, computes outcome-agreer set, plots the distribution
of pairwise Jaccard overlap on factor_support_T and factor_support_F, and
surfaces hand-inspection candidates from the low-overlap and high-overlap
ends.

Tests the primary hypothesis: among outcome-agreers, factor-support overlap
varies non-trivially. A bimodal or low-skewed distribution with low-end
mass below 0.3 supports the hypothesis; a unimodal distribution at high
overlap (median > 0.7) falsifies it.

In [ ]:
import json
from pathlib import Path
import pandas as pd

from wedge.types import Case
from wedge.metrics import is_outcome_agreer, pairwise_factor_support_overlap

RUN_JSONL = Path("runs/<replace-with-run-id>.jsonl")  # set to the latest run before running

cases: list[Case] = []
with RUN_JSONL.open() as f:
    for line in f:
        cases.append(Case.from_dict(json.loads(line)))

print(f"loaded {len(cases)} cases from {RUN_JSONL}")

In [ ]:
records = []
for c in cases:
    if not c.per_model:
        continue
    agreer = is_outcome_agreer(c.per_model)
    overlap_T, overlap_F = pairwise_factor_support_overlap(c.per_model)
    records.append({
        "case_id": c.case_id,
        "origin": c.origin,
        "label": c.label,
        "outcome_agreer": agreer,
        "overlap_T": overlap_T,
        "overlap_F": overlap_F,
        "mean_T": sum(p.T for p in c.per_model) / len(c.per_model),
    })
df = pd.DataFrame(records)
df.head()

In [ ]:
agreers = df[df["outcome_agreer"]]
print("Outcome-agreer count:", len(agreers))
print("Median overlap_T among agreers:", agreers["overlap_T"].median())
print("Median overlap_F among agreers:", agreers["overlap_F"].median())
agreers["overlap_T"].hist(bins=20)

In [ ]:
# Hand-inspection candidates from the low-overlap end (most surprising).
low_T = agreers.sort_values("overlap_T").head(10)
print("Lowest overlap_T outcome-agreers:")
print(low_T[["case_id", "origin", "mean_T", "overlap_T", "overlap_F"]])

In [ ]:
# Hand-inspection candidates from the high-overlap end (control).
high_T = agreers.sort_values("overlap_T", ascending=False).head(10)
print("Highest overlap_T outcome-agreers:")
print(high_T[["case_id", "origin", "mean_T", "overlap_T", "overlap_F"]])

## What to look at

1. Pick 3 cases from `low_T` (low-overlap-T outcome-agreers) and 3 from `high_T` (high-overlap-T).
2. For each, look up the case in `cases` and print each model's `factor_support_T` features.
3. Check whether low-overlap cases show genuinely distinct reasoning (different feature sets) or merely synonyms-of-the-same-signal.
4. Record observations in a separate markdown file alongside the run; do not rationalize the result post-hoc.